# Benchmark externo congelado — 1.000 RAISE + 1.000 MNW

Este notebook cria um **novo** conjunto e reavalia os checkpoints das Fases 1–3.4. Não treina, não recalibra e não apaga resultados históricos. Execute as células em ordem; os downloads e as avaliações só começam nas células acionadas manualmente. O conjunto externo não pode orientar hiperparâmetros ou seleção de modelo.

RAISE: formulário de pesquisa/ensino para obter CSV com URLs NEF. MNW: repositório público, sem token próprio; o token GitHub existente serve apenas para ler o código privado do TCC.

In [6]:
# 1. GPU e dependências
import torch
assert torch.cuda.is_available(), 'Conecte uma GPU do Colab'
print('GPU:', torch.cuda.get_device_name(0))
print('Verificando/instalando dependências...')
%pip install datasets scikit-learn cryptography

GPU: Tesla T4
Verificando/instalando dependências...


In [7]:
# 2. Drive e pastas persistentes; nenhuma pasta de execução anterior é alterada.
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for folder in ('credenciais', 'modelos', 'dados_fase3_4', 'dados_externos', 'experimentos', 'entradas', 'resultados'):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive: /content/drive/MyDrive/TCC_Final


In [8]:

# 3. Desbloqueio da credencial GitHub
# Compatível com os formatos antigo e novo de github.enc

import base64
import getpass
import json

from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

credential_path = ROOT / "credenciais" / "github.enc"

if not credential_path.is_file():
    raise FileNotFoundError(
        f"Credencial não encontrada: {credential_path}"
    )

credential = json.loads(
    credential_path.read_text(encoding="utf-8")
)

# Identifique pelo nome do campo cifrado, sem exigir metadados no formato antigo.
if not isinstance(credential, dict) or "salt" not in credential:
    raise ValueError("Formato de github.enc não reconhecido.")
if "ciphertext" in credential and "token" in credential:
    raise ValueError("Credencial ambígua: dois campos cifrados.")
if "ciphertext" in credential:
    if credential.get("version") != 1:
        raise ValueError("Versão de credencial não suportada.")
    iterations = int(credential["iterations"])
    encrypted_token = credential["ciphertext"]
elif "token" in credential:
    iterations = 600000
    encrypted_token = credential["token"]

else:
    raise ValueError(
        "Formato de github.enc não reconhecido."
    )
if iterations < 600000 or not isinstance(encrypted_token, str):
    raise ValueError("Parâmetros da credencial inválidos.")

salt = base64.b64decode(credential["salt"])

password = getpass.getpass(
    "Senha de desbloqueio do GitHub: "
)

kdf = PBKDF2HMAC(
    algorithm=hashes.SHA256(),
    length=32,
    salt=salt,
    iterations=iterations
)

key = base64.urlsafe_b64encode(
    kdf.derive(password.encode())
)

try:
    token = Fernet(key).decrypt(
        encrypted_token.encode()
    ).decode()

except InvalidToken:
    raise ValueError(
        "Senha incorreta ou credencial corrompida."
    ) from None

finally:
    del password, kdf, key

print("Credencial desbloqueada nesta sessão.")


Credencial desbloqueada nesta sessão.


In [9]:
# 4. Snapshot imutável da branch privada no runtime; não faz git clone nem toca no Drive.
import io, os, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
OWNER, REPO, BRANCH = 'vicemnovo', 'TCC_Final', 'main'
API = f'https://api.github.com/repos/vicemnovo/TCC_Final'
def github_read(url):
    request = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase3-Colab'})
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            chunks, size, last_notice = [], 0, time.monotonic()
            while chunk := response.read(1024 * 1024):
                chunks.append(chunk)
                size += len(chunk)
                if time.monotonic() - last_notice >= 30:
                    print(f'Download GitHub em andamento: {size / 1048576:.1f} MiB', flush=True)
                    last_notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code} ao consultar {url.split(API)[-1] or "/"}') from None
try:
    repo_info = json.loads(github_read(API))
except RuntimeError as error:
    raise RuntimeError('Repositório privado inacessível: verifique owner, seleção do repositório no token e permissão Metadata (read).') from error
try:
    branch_info = json.loads(github_read(f'{API}/branches/{BRANCH}'))
except RuntimeError as error:
    if 'HTTP 404' not in str(error):
        raise
    raise RuntimeError(f'Repositório acessível, mas a branch {BRANCH!r} não foi encontrada. Publique essa branch; a branch padrão é {repo_info["default_branch"]!r}.') from error
commit = branch_info['commit']['sha']
assert re.fullmatch(r'[0-9a-f]{40}', commit)
CODE_ROOT = Path('/content/tcc_codigo') / commit
required = ('Notebooks/fase_3/fase3_4/fase3_4_finetune_v2.py', 'experimentos/amostragem_fase3_4.py', 'experimentos/particoes_fase3_4.py', 'Notebooks/fase_3/fase3_3/fase3.3_compressao.ipynb')
if not CODE_ROOT.exists():
    print('Baixando snapshot do código...', flush=True)
    archive = github_read(f'{API}/zipball/{commit}')
    CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='tcc_sync_', dir=CODE_ROOT.parent) as temporary:
        staged = Path(temporary)
        print('Extraindo snapshot no runtime...', flush=True)
        with zipfile.ZipFile(io.BytesIO(archive)) as zf:
            last_notice = time.monotonic()
            extracted = 0
            for item in zf.infolist():
                parts = Path(item.filename).parts
                if item.is_dir() or len(parts) < 2:
                    continue
                relative = Path(*parts[1:])
                if relative.is_absolute() or '..' in relative.parts:
                    raise ValueError('Caminho inseguro no snapshot')
                destination = staged / relative
                destination.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(item) as source, destination.open('wb') as output:
                    shutil.copyfileobj(source, output)
                extracted += 1
                if time.monotonic() - last_notice >= 30:
                    print(f'Extração em andamento: {extracted} arquivos', flush=True)
                    last_notice = time.monotonic()
        if not all((staged / name).is_file() for name in required):
            raise FileNotFoundError('Publique a branch com o script e o notebook original antes de sincronizar')
        staged.rename(CODE_ROOT)
assert all((CODE_ROOT / name).is_file() for name in required)
del token
print('Código sincronizado:', commit)
print('Pasta local do runtime:', CODE_ROOT)

Código sincronizado: 237e509afe90e1a199614a9f02727a5602092dfa
Pasta local do runtime: /content/tcc_codigo/237e509afe90e1a199614a9f02727a5602092dfa


## Fontes e dependências

Na [página oficial RAISE](https://loki.disi.unitn.it/RAISE/download.html), escolha RAISE-1k, preencha nome, afiliação e e-mail, aceite os termos e baixe o CSV de metadados/URLs. Salve-o uma única vez como `/content/drive/MyDrive/TCC_Final/entradas/RAISE_1k.csv`. Não coloque o CSV nem as imagens no GitHub. A MNW é pública, mas os arquivos Git LFS serão verificados pelo piloto.

In [10]:
# 5. Dependências e entradas; sem download de imagens.
import importlib.util, subprocess, sys
if importlib.util.find_spec('rawpy') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps', 'rawpy'])
import subprocess, sys, time, threading, queue, shutil
BENCH = CODE_ROOT / 'experimentos/benchmark_2000.py'
EVAL = CODE_ROOT / 'experimentos/avaliar_benchmark_2000.py'
for path in (BENCH, EVAL):
    if not path.is_file():
        raise FileNotFoundError(f'Código novo ainda não publicado na branch: {path}')
RAISE_CSV = ROOT / 'entradas/RAISE_1k.csv'
if not RAISE_CSV.is_file():
    raise FileNotFoundError(f'Obtenha o CSV pelo formulário RAISE e salve-o em {RAISE_CSV}')
print('Código:', CODE_ROOT)
print('CSV RAISE encontrado:', RAISE_CSV)
print('Espaço livre /content: %.1f GB' % (shutil.disk_usage('/content').free / 1e9))
print('Espaço reportado pelo Drive: %.1f GB (confira também a quota da conta)' % (shutil.disk_usage(ROOT).free / 1e9))

def run_live(label, command):
    print(f'{label}: iniciando', flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    lines = queue.Queue()
    def read_output():
        for line in process.stdout:
            lines.put(line)
        lines.put(None)
    threading.Thread(target=read_output, daemon=True).start()
    begin = time.monotonic()
    last = begin
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None:
                break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: processo ativo há {(time.monotonic()-begin)/60:.1f} min; sem nova saída', flush=True)
                last = time.monotonic()
    result = process.wait()
    if result:
        raise RuntimeError(f'{label} falhou com código {result}; veja a saída acima')
    print(f'{label}: concluído em {(time.monotonic()-begin)/60:.1f} min', flush=True)

Código: /content/tcc_codigo/237e509afe90e1a199614a9f02727a5602092dfa
CSV RAISE encontrado: /content/drive/MyDrive/TCC_Final/entradas/RAISE_1k.csv
Espaço livre /content: 207.3 GB
Espaço reportado pelo Drive: 8.2 GB (confira também a quota da conta)


## Piloto e congelamento

Execute o piloto primeiro. Ele baixa **uma** imagem de cada fonte para testar URL, Git LFS, RAW e conversão. Depois, execute a preparação completa **uma única vez**; ela fixa a seleção antes da inferência, retoma downloads interrompidos e só publica o manifesto quando as 2.000 imagens estiverem íntegras. Os RAWs são temporários em /content; apenas JPEGs normalizados e hashes ficam no Drive. Não execute uma avaliação enquanto a pasta `.incompleto` existir.

In [6]:
# 6. PILOTO MANUAL: 1 RAISE + 1 MNW; não cria benchmark final.
run_live('piloto', [sys.executable, '-u', str(BENCH), 'pilot', '--root', str(ROOT), '--raise-csv', str(RAISE_CSV)])

In [14]:
# 7. PREPARAÇÃO MANUAL: baixa e congela 1.000 RAISE + 1.000 MNW.
# Pode demorar; se desconectar, execute de novo para retomar o plano já persistido.
run_live('preparação 2000', [sys.executable, '-u', str(BENCH), 'prepare', '--root', str(ROOT), '--raise-csv', str(RAISE_CSV)])

In [9]:
# 7A. Diagnóstico somente-leitura após falha de integridade; não modifica o plano.
from collections import defaultdict
work = ROOT / 'dados_externos/benchmark_externo_raise_mnw_2000_v1.incompleto'
plan = json.loads((work / 'plan.json').read_text(encoding='utf-8'))
hashes = json.loads((work / 'hashes.json').read_text(encoding='utf-8'))
expected = {key for row in plan['rows'] for key in ('source:' + row['id'], row['image'])}
print('Linhas no plano:', len(plan['rows']))
print('Hashes esperados/registrados/distintos:', len(expected), len(hashes), len(set(hashes.values())))
print('Chaves faltantes:', sorted(expected - hashes.keys())[:20])
print('Chaves extras:', sorted(hashes.keys() - expected)[:20])
by_hash = defaultdict(list)
for row in plan['rows']:
    for kind, key in (('original', 'source:' + row['id']), ('jpeg', row['image'])):
        if key in hashes:
            by_hash[hashes[key]].append((kind, row['label'], row['group'], row['id']))
duplicates = [items for items in by_hash.values() if len(items) > 1]
print('Grupos de conteúdo duplicado:', len(duplicates))
for index, items in enumerate(duplicates[:30], 1):
    print(f'{index}:', items)
print('Nenhum arquivo foi alterado por este diagnóstico.')

### Reparo de duplicatas reais antes de congelar (somente se a auditoria acusar pares RAISE)

Obtenha o CSV de metadados **RAISE-2k** pela fonte oficial e salve-o em `TCC_Final/entradas/RAISE_2k.csv`. Não baixe o pacote de imagens. Após publicar a correção na branch, execute novamente as células **3, 4 e 5**: a célula 4 apaga o token da memória após sincronizar, portanto a 3 precisa desbloqueá-lo novamente. A célula 7B verifica candidatas extras por hash e cria `plan_repaired.json`; não altera `plan.json` nem os JPEGs existentes. Em seguida execute novamente a célula 7 para baixar somente as substitutas e concluir a auditoria. Não execute a célula 8 antes de aparecer `Benchmark congelado`.

In [13]:
# 7B. Reparo MANUAL e versionado dos seis pares RAW/JPEG reais duplicados.
extra_csv = ROOT / 'entradas/RAISE_2k.csv'
if not extra_csv.is_file():
    raise FileNotFoundError(f'Obtenha somente o CSV de metadados RAISE-2k: {extra_csv}')
if 'repair_raise_duplicates' not in BENCH.read_text(encoding='utf-8'):
    raise RuntimeError('Código ainda antigo no runtime; publique a correção e reexecute as células 3, 4 e 5')
run_live('reparo RAISE', [sys.executable, '-u', str(BENCH), 'repair',
                           '--root', str(ROOT), '--raise-extra-csv', str(extra_csv)])

In [11]:
# 8. Cópia/verificação em /content a cada sessão. Não lê imagens do Drive durante inferência.
run_live('cópia local', [sys.executable, '-u', str(BENCH), 'stage', '--root', str(ROOT)])

cópia local: iniciando
Cópia/verificação: 50/2000
Cópia/verificação: 100/2000
Cópia/verificação: 150/2000
Cópia/verificação: 200/2000
Cópia/verificação: 250/2000
Cópia/verificação: 300/2000
Cópia/verificação: 350/2000
Cópia/verificação: 400/2000
Cópia/verificação: 450/2000
Cópia/verificação: 500/2000
Cópia/verificação: 550/2000
Cópia/verificação: 600/2000
Cópia/verificação: 650/2000
Cópia/verificação: 700/2000
Cópia/verificação: 750/2000
Cópia/verificação: 800/2000
Cópia/verificação: 850/2000
Cópia/verificação: 900/2000
Cópia/verificação: 950/2000
Cópia/verificação: 1000/2000
Cópia/verificação: 1050/2000
Cópia/verificação: 1100/2000
Cópia/verificação: 1150/2000
Cópia/verificação: 1200/2000
Cópia/verificação: 1250/2000
Cópia/verificação: 1300/2000
Cópia/verificação: 1350/2000
Cópia/verificação: 1400/2000
Cópia/verificação: 1450/2000
Cópia/verificação: 1500/2000
Cópia/verificação: 1550/2000
Cópia/verificação: 1600/2000
Cópia/verificação: 1650/2000
Cópia/verificação: 1700/2000
Cópia/verif

## Nova comparação histórica no mesmo conjunto

As células abaixo são **manuais** e produzem resultados novos em `resultados/benchmark_externo_raise_mnw_2000_v1/<fase>/`. Elas nunca reescrevem métricas originais. Fase 1 usa seu checkpoint sem temperatura persistida (T=1, limiar=0,5); Fase 2 usa resize 224; Fases 3 usam cinco recortes. Pesos, temperatura e limiar das Fases 2–3 vêm de cada checkpoint. O conjunto externo não deve orientar novo treino, escolha de modelo ou limiar.

**Retomada após a falha da Fase 1:** o snapshot `30cf718` procurava `Phase1Detector` no notebook da Fase 3.3 e falhou antes de salvar métricas. Antes de repetir `avaliar('fase1')`, sincronize um commit posterior com a correção em `experimentos/avaliar_benchmark_2000.py` (células 3–5 do notebook). Se o runtime foi reiniciado, execute também a cópia/verificação e a conferência dos checkpoints. Não repita piloto, reparo ou preparo do conjunto congelado.

In [12]:
# 9. Conferência dos checkpoints; faltantes são exibidos sem modificar nada.
MODELS = {'fase1':'phase1_best_model.pt', 'fase2':'fase2_gate_vit_fft_cnn/phase2_final.pt',
          'fase3_1':'fase3_multigerador/phase3_final.pt',
          'fase3_2':'fase3_holdout_v2/phase3_final.pt',
          'fase3_3':'fase3_3_compressao_v1/phase3_final.pt',
          'fase3_4_v1':'fase3_4_finetune_v1/phase3_final.pt',
          'fase3_4_v2':'fase3_4_finetune_v2_sqrt_sampler/phase3_final.pt'}
for name, rel in MODELS.items():
    path = ROOT / 'modelos' / rel
    print(f'{name}: '+(f'presente ({path.stat().st_size:,} bytes)' if path.is_file() else 'AUSENTE'))

def avaliar(name):
    if name not in MODELS:
        raise ValueError(name)
    run_live(name, [sys.executable, '-u', str(EVAL), name, '--root', str(ROOT),
                    '--code-root', str(CODE_ROOT)])

fase1: presente (22,460,755 bytes)
fase2: presente (40,626,462 bytes)
fase3_1: presente (40,627,742 bytes)
fase3_2: presente (40,627,806 bytes)
fase3_3: presente (40,627,806 bytes)
fase3_4_v1: presente (40,628,062 bytes)
fase3_4_v2: presente (40,628,318 bytes)


In [13]:
# Avaliação MANUAL: Fase 1. Se já houver resultado, preserva e recusa sobrescrever.
avaliar('fase1')

fase1: iniciando
Integridade antes da avaliação: 100/2000
Integridade antes da avaliação: 200/2000
Integridade antes da avaliação: 300/2000
Integridade antes da avaliação: 400/2000
Integridade antes da avaliação: 500/2000
Integridade antes da avaliação: 600/2000
Integridade antes da avaliação: 700/2000
Integridade antes da avaliação: 800/2000
Integridade antes da avaliação: 900/2000
Integridade antes da avaliação: 1000/2000
Integridade antes da avaliação: 1100/2000
Integridade antes da avaliação: 1200/2000
Integridade antes da avaliação: 1300/2000
Integridade antes da avaliação: 1400/2000
Integridade antes da avaliação: 1500/2000
Integridade antes da avaliação: 1600/2000
Integridade antes da avaliação: 1700/2000
Integridade antes da avaliação: 1800/2000
Integridade antes da avaliação: 1900/2000
Integridade antes da avaliação: 2000/2000
Dispositivo: cuda
fase1: checkpoint compatível carregado | T=1.000000 | limiar=0.500000
fase1: 100/2000 inferências
fase1: 200/2000 inferências
fase1: 3

In [14]:
# Avaliação MANUAL: Fase 2. Se já houver resultado, preserva e recusa sobrescrever.
avaliar('fase2')

fase2: iniciando
Integridade antes da avaliação: 100/2000
Integridade antes da avaliação: 200/2000
Integridade antes da avaliação: 300/2000
Integridade antes da avaliação: 400/2000
Integridade antes da avaliação: 500/2000
Integridade antes da avaliação: 600/2000
Integridade antes da avaliação: 700/2000
Integridade antes da avaliação: 800/2000
Integridade antes da avaliação: 900/2000
Integridade antes da avaliação: 1000/2000
Integridade antes da avaliação: 1100/2000
Integridade antes da avaliação: 1200/2000
Integridade antes da avaliação: 1300/2000
Integridade antes da avaliação: 1400/2000
Integridade antes da avaliação: 1500/2000
Integridade antes da avaliação: 1600/2000
Integridade antes da avaliação: 1700/2000
Integridade antes da avaliação: 1800/2000
Integridade antes da avaliação: 1900/2000
Integridade antes da avaliação: 2000/2000
Dispositivo: cuda
fase2: checkpoint compatível carregado | T=0.390912 | limiar=0.725000
fase2: 100/2000 inferências
fase2: 200/2000 inferências
fase2: 3

In [15]:
# Avaliação MANUAL: Fase 3.1. Se já houver resultado, preserva e recusa sobrescrever.
avaliar('fase3_1')

fase3_1: iniciando
Integridade antes da avaliação: 100/2000
Integridade antes da avaliação: 200/2000
Integridade antes da avaliação: 300/2000
Integridade antes da avaliação: 400/2000
Integridade antes da avaliação: 500/2000
Integridade antes da avaliação: 600/2000
Integridade antes da avaliação: 700/2000
Integridade antes da avaliação: 800/2000
Integridade antes da avaliação: 900/2000
Integridade antes da avaliação: 1000/2000
Integridade antes da avaliação: 1100/2000
Integridade antes da avaliação: 1200/2000
Integridade antes da avaliação: 1300/2000
Integridade antes da avaliação: 1400/2000
Integridade antes da avaliação: 1500/2000
Integridade antes da avaliação: 1600/2000
Integridade antes da avaliação: 1700/2000
Integridade antes da avaliação: 1800/2000
Integridade antes da avaliação: 1900/2000
Integridade antes da avaliação: 2000/2000
Dispositivo: cuda
fase3_1: checkpoint compatível carregado | T=0.795099 | limiar=0.310000
fase3_1: 100/2000 inferências
fase3_1: 200/2000 inferências


In [16]:
# Avaliação MANUAL: Fase 3.2. Se já houver resultado, preserva e recusa sobrescrever.
avaliar('fase3_2')

fase3_2: iniciando
Integridade antes da avaliação: 100/2000
Integridade antes da avaliação: 200/2000
Integridade antes da avaliação: 300/2000
Integridade antes da avaliação: 400/2000
Integridade antes da avaliação: 500/2000
Integridade antes da avaliação: 600/2000
Integridade antes da avaliação: 700/2000
Integridade antes da avaliação: 800/2000
Integridade antes da avaliação: 900/2000
Integridade antes da avaliação: 1000/2000
Integridade antes da avaliação: 1100/2000
Integridade antes da avaliação: 1200/2000
Integridade antes da avaliação: 1300/2000
Integridade antes da avaliação: 1400/2000
Integridade antes da avaliação: 1500/2000
Integridade antes da avaliação: 1600/2000
Integridade antes da avaliação: 1700/2000
Integridade antes da avaliação: 1800/2000
Integridade antes da avaliação: 1900/2000
Integridade antes da avaliação: 2000/2000
Dispositivo: cuda
fase3_2: checkpoint compatível carregado | T=0.651368 | limiar=0.305000
fase3_2: 100/2000 inferências
fase3_2: 200/2000 inferências


In [17]:
# Avaliação MANUAL: Fase 3.3. Se já houver resultado, preserva e recusa sobrescrever.
avaliar('fase3_3')

fase3_3: iniciando
Integridade antes da avaliação: 100/2000
Integridade antes da avaliação: 200/2000
Integridade antes da avaliação: 300/2000
Integridade antes da avaliação: 400/2000
Integridade antes da avaliação: 500/2000
Integridade antes da avaliação: 600/2000
Integridade antes da avaliação: 700/2000
Integridade antes da avaliação: 800/2000
Integridade antes da avaliação: 900/2000
Integridade antes da avaliação: 1000/2000
Integridade antes da avaliação: 1100/2000
Integridade antes da avaliação: 1200/2000
Integridade antes da avaliação: 1300/2000
Integridade antes da avaliação: 1400/2000
Integridade antes da avaliação: 1500/2000
Integridade antes da avaliação: 1600/2000
Integridade antes da avaliação: 1700/2000
Integridade antes da avaliação: 1800/2000
Integridade antes da avaliação: 1900/2000
Integridade antes da avaliação: 2000/2000
Dispositivo: cuda
fase3_3: checkpoint compatível carregado | T=0.650134 | limiar=0.470000
fase3_3: 100/2000 inferências
fase3_3: 200/2000 inferências


In [18]:
# Avaliação MANUAL: Fase 3.4 v1. Se já houver resultado, preserva e recusa sobrescrever.
avaliar('fase3_4_v1')

fase3_4_v1: iniciando
Integridade antes da avaliação: 100/2000
Integridade antes da avaliação: 200/2000
Integridade antes da avaliação: 300/2000
Integridade antes da avaliação: 400/2000
Integridade antes da avaliação: 500/2000
Integridade antes da avaliação: 600/2000
Integridade antes da avaliação: 700/2000
Integridade antes da avaliação: 800/2000
Integridade antes da avaliação: 900/2000
Integridade antes da avaliação: 1000/2000
Integridade antes da avaliação: 1100/2000
Integridade antes da avaliação: 1200/2000
Integridade antes da avaliação: 1300/2000
Integridade antes da avaliação: 1400/2000
Integridade antes da avaliação: 1500/2000
Integridade antes da avaliação: 1600/2000
Integridade antes da avaliação: 1700/2000
Integridade antes da avaliação: 1800/2000
Integridade antes da avaliação: 1900/2000
Integridade antes da avaliação: 2000/2000
Dispositivo: cuda
fase3_4_v1: checkpoint compatível carregado | T=0.457128 | limiar=0.460000
fase3_4_v1: 100/2000 inferências
fase3_4_v1: 200/2000 

In [19]:
# Avaliação MANUAL: Fase 3.4 v2. Se já houver resultado, preserva e recusa sobrescrever.
avaliar('fase3_4_v2')

fase3_4_v2: iniciando
Integridade antes da avaliação: 100/2000
Integridade antes da avaliação: 200/2000
Integridade antes da avaliação: 300/2000
Integridade antes da avaliação: 400/2000
Integridade antes da avaliação: 500/2000
Integridade antes da avaliação: 600/2000
Integridade antes da avaliação: 700/2000
Integridade antes da avaliação: 800/2000
Integridade antes da avaliação: 900/2000
Integridade antes da avaliação: 1000/2000
Integridade antes da avaliação: 1100/2000
Integridade antes da avaliação: 1200/2000
Integridade antes da avaliação: 1300/2000
Integridade antes da avaliação: 1400/2000
Integridade antes da avaliação: 1500/2000
Integridade antes da avaliação: 1600/2000
Integridade antes da avaliação: 1700/2000
Integridade antes da avaliação: 1800/2000
Integridade antes da avaliação: 1900/2000
Integridade antes da avaliação: 2000/2000
Dispositivo: cuda
fase3_4_v2: checkpoint compatível carregado | T=0.440257 | limiar=0.370000
fase3_4_v2: 100/2000 inferências
fase3_4_v2: 200/2000 

In [20]:
# 17. Resumo somente-leitura dos resultados novos; não reavalia nem modifica testes.
import json
base = ROOT / 'resultados/benchmark_externo_raise_mnw_2000_v1'
for name in MODELS:
    path = base / name / 'metrics.json'
    if path.is_file():
        m = json.loads(path.read_text(encoding='utf-8'))
        print(f"{name}: acurácia={m['accuracy']:.3f} AUC={m['auc']:.3f} recall={m['recall']:.3f} especificidade={m['specificity']:.3f} | manifesto={m['manifest_sha256'][:12]}")
    else:
        print(f'{name}: ainda não avaliado')

fase1: acurácia=0.502 AUC=0.542 recall=0.006 especificidade=0.998 | manifesto=2024b501e2c4
fase2: acurácia=0.500 AUC=0.643 recall=0.000 especificidade=1.000 | manifesto=2024b501e2c4
fase3_1: acurácia=0.550 AUC=0.557 recall=0.317 especificidade=0.784 | manifesto=2024b501e2c4
fase3_2: acurácia=0.566 AUC=0.605 recall=0.288 especificidade=0.843 | manifesto=2024b501e2c4
fase3_3: acurácia=0.600 AUC=0.738 recall=0.226 especificidade=0.973 | manifesto=2024b501e2c4
fase3_4_v1: acurácia=0.592 AUC=0.736 recall=0.207 especificidade=0.977 | manifesto=2024b501e2c4
fase3_4_v2: acurácia=0.621 AUC=0.738 recall=0.289 especificidade=0.953 | manifesto=2024b501e2c4
